In [4]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Psychophysical Kernel Analysis (Motion-Energy Based)

Expands on `2.13-psychophysical-kernal.ipynb` and `2.11-me-analysis.ipynb`. Goals:

1. Compute a rigorous, **choice-conditioned, coherence-mean-subtracted** psychophysical kernel (the noise-residual kernel of Nienborg & Cumming 2009 / Okazawa et al. 2018), not just the outcome-conditioned kernel on raw motion energy used in `2.13`.
2. Build the kernel hierarchically: trial -> session -> mouse -> group, so each of the mice contributes one data point to group statistics (not each of the ~25k trials pooled together).
3. Add real inferential statistics: bootstrap CI across mice + a cluster-mass permutation test for time windows where the kernel differs from zero.
4. Directly re-derive the **primacy effect** from ground-truth motion energy: pulse-triggered, choice-conditioned motion energy, split by pulse-timing bin (early/mid/late) x direction, to confirm -- using measured stimulus rather than nominal coherence/pulse bins -- that the choice-relevant impact of a pulse decays with latency (complements the pooled GLM in `1.0-psychometric-analysis.ipynb`).
5. Check replication across the two independent rigs/cohorts (UCLA vs UW, see `0.0-data-compilation.ipynb`).

**This notebook has not been executed** -- the data directory is not mounted in the environment it was written in. Code follows the schema used in `2.10`-`2.13` (`valid_data.csv`, `motion_energies_normalized/me_{type}.pkl`); run top-to-bottom on a machine with data mounted and sanity-check intermediate shapes/values before trusting the figures. See the caveats section at the end for specific assumptions to verify.

## Setup

In [5]:
from pathlib import Path
import pickle
import warnings

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

from notebooks.imports import *
from config import dir_config
from src.utils import scientific_plotting as spu

warnings.filterwarnings("ignore", category=UserWarning)
spu.setup()

raw_dir = Path(dir_config.data.raw)
processed_dir = Path(dir_config.data.processed)

# Figures for the manuscript/poster -- path relative to this notebook's location (notebooks/),
# not dir_config.project_dir (which resolves to the in-container "/src" path, not this repo checkout).
FIGURE_DIR = Path("../dissemination/figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

me_type = "global"  # "global", "center", or a quadrant key -- see notebooks/2.10-motion-energy-postprocessing.ipynb
RNG = np.random.default_rng(0)

## Load data

In [7]:
with open(Path(processed_dir, f"motion_energies_normalized/me_{me_type}.pkl"), "rb") as f:
    motion_energy_dict = pickle.load(f)

data = pd.read_csv(Path(processed_dir, "valid_data.csv"), index_col=None)
data["reaction_time"] = (data["reaction_time"] * 1000).astype(int)  # seconds -> ms
data = data[data["reaction_time"] < 3000].reset_index(drop=True)

# Two independent rigs/cohorts, encoded in the subject_id prefix (see notebooks/0.0-data-compilation.ipynb:
# "Data is collected from two different set of mice. 1. UCLA: Big screen, 2 pulse times 2. UW: smaller screen, 3 pulse times")
data["cohort"] = np.where(data["subject_id"].str.startswith("PSUIM"), "UCLA", "UW")

valid_keys = set(data["agg_code"].values)
motion_energy_dict = {k: v for k, v in motion_energy_dict.items() if k in valid_keys}
motion_energy_array = np.array([motion_energy_dict[k] for k in data["agg_code"].values])
assert len(motion_energy_array) == len(data), f"Length mismatch: {len(motion_energy_array)} vs {len(data)}"

# Stimulus/motion-energy frame rate. NOTE: UCLA rig runs at 190Hz and UW at 60Hz (scripts/ucla_rdk.py,
# scripts/uw_rdk.py) -- confirm whether `me_{type}.pkl` traces were resampled to a common rate during
# notebooks/2.10 postprocessing before assuming a single FRAME_MS applies to both cohorts.
FRAME_MS = 1000 / 60

print(f"{data['subject_id'].nunique()} mice, {data['idx_session'].nunique()} sessions, {len(data)} trials")
data.groupby("cohort")["subject_id"].nunique()

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/processed/motion_energies_normalized/me_global.pkl'

## Pulse-condition bookkeeping

In [ ]:
pulse_mapping = {
    -3: "late_left_pulse", -2: "middle_left_pulse", -1: "early_left_pulse",
    0: "control",
    1: "early_right_pulse", 2: "middle_right_pulse", 3: "late_right_pulse",
}
pulse_time_bin_map = {1: "early", 2: "middle", 3: "late"}  # by |pulse_type|
pulse_dir_map = {-1: "left", 1: "right"}  # by sign(pulse_type)

data["pulse_condition"] = data["pulse_type"].map(pulse_mapping)
data["pulse_time_bin"] = data["pulse_type"].abs().map(pulse_time_bin_map)
data["pulse_direction"] = np.sign(data["pulse_type"]).map(pulse_dir_map)

## 1. Choice-conditioned, coherence-mean-subtracted kernel

The raw psychophysical kernel (mean ME for right choices minus mean ME for left choices, as in `2.13`) is
confounded by nominal coherence: the physical stimulus already differs systematically between rightward-
and leftward-choice trials whenever `|coherence| > 0`, since higher coherence drives both more motion
energy in that direction *and* more choices in that direction. Following Nienborg & Cumming (2009) /
Okazawa et al. (2018), we instead:

1. Compute, per `|signed_coherence|` condition (within a session), the mean ME trace across all trials of
   that condition -- the "expected" stimulus.
2. Subtract it from each trial's ME trace to get a noise residual that is, by construction, uncorrelated
   with nominal coherence.
3. Average residuals split by choice -- the resulting kernel reflects only the stimulus fluctuations that
   covary with choice, not the average signal strength.

We use pulse-free (`pulse_type == 0`) trials only, so a genuine pulse never contaminates this baseline
kernel. Both the raw and residual kernels are computed below so they can be compared directly.

In [ ]:
def right_align_rows_with_nan_padding(arr):
    result = np.full_like(arr, np.nan, dtype=float)
    for i, row in enumerate(arr):
        valid = row[~np.isnan(row)]
        if len(valid):
            result[i, -len(valid):] = valid
    return result


def trim_by_nan_fraction(arr, threshold=0.4, from_end=False):
    """Drop leading/trailing time bins where more than `threshold` of rows are NaN.

    from_end=False trims the tail (for stimulus-onset-aligned data, where NaNs accumulate
    late as shorter-RT trials run out of frames). from_end=True trims the head (for
    response-onset-aligned / right-aligned data, where NaNs accumulate early).
    """
    nan_ratio = np.isnan(arr).mean(axis=0)
    over = np.where(nan_ratio > threshold)[0]
    if len(over) == 0:
        return arr
    cut = over[-1] + 1 if from_end else over[0]
    return arr[:, cut:] if from_end else arr[:, :cut]


def residual_motion_energy(me, signed_coherence):
    """Subtract, per |coherence| condition, the condition-mean ME trace from each trial."""
    residual = np.full_like(me, np.nan, dtype=float)
    for coh in np.unique(signed_coherence):
        mask = signed_coherence == coh
        cond_mean = np.nanmean(me[mask], axis=0)
        residual[mask] = me[mask] - cond_mean
    return residual


def session_kernel(me, choice, alignment="stimulus_onset", nan_threshold=0.5):
    """Right-choice minus left-choice mean ME (or residual-ME) trace for one session.

    NOTE: assumes `choice` is coded 1 = rightward, 0 = leftward -- verify against the
    schema produced in notebooks/0.1-preprocessing-data.ipynb before trusting kernel sign.
    """
    me = me.copy()
    drop_cols = np.isnan(me).mean(axis=0) > nan_threshold
    me[:, drop_cols] = np.nan
    if alignment == "response_onset":
        me = right_align_rows_with_nan_padding(me)
    right = me[choice == 1]
    left = me[choice == 0]
    if len(right) == 0 or len(left) == 0:
        return np.full(me.shape[1], np.nan)
    return np.nanmean(right, axis=0) - np.nanmean(left, axis=0)

In [ ]:
def build_hierarchical_kernels(df, me_array, use_residual, alignment, nan_threshold=0.5, trim_threshold=0.4):
    """session -> mouse -> group kernel, so group stats treat each mouse as one sample."""
    session_rows = []
    for idx in df["idx_session"].unique():
        mask = (df["idx_session"] == idx).values & (df["pulse_type"] == 0).values
        if mask.sum() < 10:
            continue
        sub = df.loc[mask]
        me = me_array[mask]
        if use_residual:
            me = residual_motion_energy(me, sub["signed_coherence"].values)
        kernel = session_kernel(me, sub["choice"].values, alignment=alignment, nan_threshold=nan_threshold)
        session_rows.append({
            "idx_session": idx,
            "subject_id": sub["subject_id"].iloc[0],
            "cohort": sub["cohort"].iloc[0],
            "kernel": kernel,
        })

    session_df = pd.DataFrame(session_rows)

    max_len = max(len(k) for k in session_df["kernel"])

    def pad(k):
        if alignment == "response_onset":
            return np.pad(k, (max_len - len(k), 0), constant_values=np.nan)
        return np.pad(k, (0, max_len - len(k)), constant_values=np.nan)

    session_mat = np.vstack(session_df["kernel"].apply(pad).values)

    mouse_rows = []
    session_df = session_df.assign(row=range(len(session_df)))
    for subj, sub in session_df.groupby("subject_id"):
        rows = sub["row"].values
        mouse_rows.append({
            "subject_id": subj,
            "cohort": sub["cohort"].iloc[0],
            "kernel": np.nanmean(session_mat[rows], axis=0),
        })
    mouse_df = pd.DataFrame(mouse_rows)
    mouse_mat = np.vstack(mouse_df["kernel"].values)
    mouse_mat = trim_by_nan_fraction(mouse_mat, threshold=trim_threshold, from_end=(alignment == "response_onset"))

    return session_df, session_mat, mouse_df, mouse_mat


kernels = {}
for use_residual in (False, True):
    for alignment in ("stimulus_onset", "response_onset"):
        key = ("residual" if use_residual else "raw", alignment)
        kernels[key] = build_hierarchical_kernels(data, motion_energy_array, use_residual, alignment)

{k: v[3].shape for k, v in kernels.items()}

## 2. Group-level statistics: bootstrap CI + cluster-mass permutation test

Each mouse contributes exactly one kernel (averaged over its own sessions), so N = number of mice is the
correct sample size for inference here -- not the trial count. We bootstrap across mice for a 95% CI band,
and run a cluster-mass sign-flip permutation test to find contiguous time windows where the group kernel
differs significantly from zero, correcting for the multiple-comparisons problem across time points. This
is a minimal from-scratch implementation with no new dependency; swap in
`mne.stats.permutation_cluster_1samp_test` for a more standard/citable version if `mne` is an acceptable
addition to `pyproject.toml`.

In [ ]:
def bootstrap_ci(mouse_mat, n_boot=5000, ci=95, rng=RNG):
    n = mouse_mat.shape[0]
    boot_means = np.full((n_boot, mouse_mat.shape[1]), np.nan)
    for b in range(n_boot):
        sample = rng.integers(0, n, n)
        boot_means[b] = np.nanmean(mouse_mat[sample], axis=0)
    lo, hi = np.nanpercentile(boot_means, [(100 - ci) / 2, 100 - (100 - ci) / 2], axis=0)
    return lo, hi


def cluster_permutation_test(mouse_mat, n_perm=2000, alpha=0.05, rng=RNG):
    """Sign-flip cluster-mass permutation test of the mouse-level kernel against zero."""
    n, t = mouse_mat.shape
    crit_t = stats.t.ppf(1 - alpha / 2, n - 1)

    def cluster_masses(tvals):
        sig = np.abs(tvals) > crit_t
        masses, run_start = [], None
        for i, s in enumerate(np.append(sig, False)):
            if s and run_start is None:
                run_start = i
            elif not s and run_start is not None:
                masses.append((run_start, i, np.nansum(np.abs(tvals[run_start:i]))))
                run_start = None
        return masses

    obs_t, _ = stats.ttest_1samp(mouse_mat, 0, axis=0, nan_policy="omit")
    obs_clusters = cluster_masses(obs_t)

    signs = rng.choice([-1, 1], size=(n_perm, n))
    max_null = np.zeros(n_perm)
    for p in range(n_perm):
        t_perm, _ = stats.ttest_1samp(mouse_mat * signs[p][:, None], 0, axis=0, nan_policy="omit")
        masses = cluster_masses(t_perm)
        max_null[p] = max((m for *_, m in masses), default=0)

    sig_clusters = []
    for s, e, m in obs_clusters:
        p_val = (np.sum(max_null >= m) + 1) / (n_perm + 1)
        if p_val < alpha:
            sig_clusters.append((s, e, p_val))
    return obs_t, sig_clusters

## 3. Plot: raw vs. residual kernel, stimulus- and response-aligned

In [ ]:
def time_axis(n, alignment, frame_ms=FRAME_MS):
    t = np.arange(n) * frame_ms
    return t if alignment == "stimulus_onset" else t - t[-1]


def plot_kernel_panel(ax, mouse_mat, alignment, title, color="#1a8fe3"):
    t = time_axis(mouse_mat.shape[1], alignment)
    lo, hi = bootstrap_ci(mouse_mat)
    mean = np.nanmean(mouse_mat, axis=0)
    _, sig_clusters = cluster_permutation_test(mouse_mat)

    ax.plot(t, mouse_mat.T, color=color, alpha=0.15, lw=1)
    ax.plot(t, mean, color=color, lw=2.5, label="group mean")
    ax.fill_between(t, lo, hi, color=color, alpha=0.2, label="bootstrap 95% CI")
    for s, e, p in sig_clusters:
        ax.axvspan(t[s], t[min(e, len(t) - 1)], color="grey", alpha=0.15)
    ax.axhline(0, color="k", lw=0.8, ls="--")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(f"Time from {'stimulus' if alignment == 'stimulus_onset' else 'response'} onset (ms)")
    ax.set_ylabel("Kernel (a.u.)")
    ax.legend(fontsize=8)


fig, axs = plt.subplots(2, 2, figsize=(spu.TEXTWIDTH * 1.4, spu.TEXTWIDTH), sharey="row")
for row, use_residual in enumerate((False, True)):
    for col, alignment in enumerate(("stimulus_onset", "response_onset")):
        _, _, _, mouse_mat = kernels[("residual" if use_residual else "raw", alignment)]
        label = "Residual (coherence-subtracted)" if use_residual else "Raw"
        plot_kernel_panel(axs[row, col], mouse_mat, alignment, f"{label} kernel -- {alignment.replace('_', ' ')}")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "psychophysical_kernel_raw_vs_residual.png", dpi=300)
plt.show()

## 4. Cohort replication check (UCLA vs UW)

In [ ]:
_, _, mouse_df, mouse_mat = kernels[("residual", "stimulus_onset")]

fig, ax = plt.subplots(figsize=(spu.TEXTWIDTH, spu.TEXTWIDTH * 0.6))
for cohort, color in zip(("UCLA", "UW"), ("#d11149", "#1ccd6a")):
    sub_mat = mouse_mat[mouse_df["cohort"].values == cohort]
    plot_kernel_panel(ax, sub_mat, "stimulus_onset", "Residual kernel by cohort", color=color)
plt.tight_layout()
plt.show()

# Simple two-group check on a scalar summary (peak |kernel| in the first 300ms) -- n is small (~5
# mice/cohort) so treat this as a replication sanity check, not a well-powered equivalence test.
window = slice(0, int(300 / FRAME_MS))
amp = np.nanmax(np.abs(mouse_mat[:, window]), axis=1)
ucla_amp = amp[mouse_df["cohort"].values == "UCLA"]
uw_amp = amp[mouse_df["cohort"].values == "UW"]
u_stat, p_val = stats.mannwhitneyu(ucla_amp, uw_amp)
print(f"UCLA (n={len(ucla_amp)}) mean amp={ucla_amp.mean():.3f}, UW (n={len(uw_amp)}) mean amp={uw_amp.mean():.3f}, Mann-Whitney p={p_val:.3f}")

## 5. Pulse-triggered kernel by timing bin: re-deriving the primacy effect from measured motion energy

The pooled GLM in `notebooks/1.0-psychometric-analysis.ipynb` shows the pulse's effect on choice bias
(`alpha`) shrinks with nominal pulse latency (early/mid/late bins). Here we ask the same question directly
from ground-truth stimulus energy: is the *choice-relevant* (residual, choice-conditioned) motion energy
triggered on the pulse itself smaller in amplitude for later pulses? This needs no assumption about how
faithfully nominal coherence/timing bins track the physical stimulus, and gives an independent line of
evidence for the primacy effect.

In [ ]:
def pulse_aligned_kernel(df, me_array, time_bin, direction, window_frames=60, min_trials_per_mouse=5):
    """window_frames=60 is ~1000ms at 60Hz -- long enough to see the kernel decay back to baseline.
    Rescale if FRAME_MS above turns out to need cohort-specific correction."""
    mask = (df["pulse_time_bin"] == time_bin).values & (df["pulse_direction"] == direction).values
    sub = df.loc[mask]
    me = me_array[mask]

    onset_frame = (sub["pulse_onset"].values * 1000 / FRAME_MS).round().astype(int)
    aligned = np.full((mask.sum(), window_frames), np.nan)
    for i, (row_me, onset) in enumerate(zip(me, onset_frame)):
        end = min(onset + window_frames, row_me.shape[0])
        if end > onset:
            aligned[i, : end - onset] = row_me[onset:end]

    residual = residual_motion_energy(aligned, sub["signed_coherence"].values)
    choice = sub["choice"].values

    rows = []
    for subj in sub["subject_id"].unique():
        m = (sub["subject_id"].values == subj)
        if m.sum() < min_trials_per_mouse:
            continue
        k = session_kernel(residual[m], choice[m], alignment="stimulus_onset", nan_threshold=0.9)
        rows.append({"subject_id": subj, "cohort": sub["cohort"].values[m][0], "kernel": k})
    return pd.DataFrame(rows)


time_bins = ["early", "middle", "late"]
pulse_kernels = {
    (tb, d): pulse_aligned_kernel(data, motion_energy_array, tb, d)
    for tb in time_bins for d in ("left", "right")
}

fig, axs = plt.subplots(1, 2, figsize=(spu.TEXTWIDTH * 1.4, spu.TEXTWIDTH * 0.6), sharey=True)
colors = {"early": "#d11149", "middle": "#e6c229", "late": "#1a8fe3"}
for direction, ax in zip(("left", "right"), axs):
    for tb in time_bins:
        m = np.vstack(pulse_kernels[(tb, direction)]["kernel"].values)
        t = np.arange(m.shape[1]) * FRAME_MS
        mean = np.nanmean(m, axis=0)
        lo, hi = bootstrap_ci(m)
        sign = 1 if direction == "right" else -1
        ax.plot(t, sign * mean, color=colors[tb], lw=2, label=f"{tb} pulse")
        ax.fill_between(t, sign * lo, sign * hi, color=colors[tb], alpha=0.15)
    ax.axhline(0, color="k", lw=0.8, ls="--")
    ax.set_title(f"{direction.capitalize()} pulses")
    ax.set_xlabel("Time from pulse onset (ms)")
axs[0].set_ylabel("Choice-conditioned residual ME\n(sign-aligned to pulse direction)")
axs[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

### Quantify the amplitude decay (mouse-level, analogous to the GLM's `alpha_3`)

In [ ]:
records = []
for (tb, d), df_k in pulse_kernels.items():
    m = np.vstack(df_k["kernel"].values)
    window = slice(0, int(200 / FRAME_MS))
    amp = np.nanmax(np.abs(m[:, window]), axis=1)
    for subj, a in zip(df_k["subject_id"], amp):
        records.append({"subject_id": subj, "pulse_time_bin": tb, "pulse_direction": d, "amplitude": a})

amp_df = pd.DataFrame(records)
amp_df["time_rank"] = amp_df["pulse_time_bin"].map({"early": 1, "middle": 2, "late": 3})

# Per-mouse slope of amplitude vs. pulse-time rank, then a group-level nonparametric test on the
# slopes (Wilcoxon signed-rank against zero) -- mirrors treating "mouse" as the unit of replication,
# same logic as the kernel statistics above, instead of pooling all trials into one regression.
slopes = []
for subj, sub in amp_df.groupby("subject_id"):
    sub_mean = sub.groupby("time_rank")["amplitude"].mean().reset_index()
    if sub_mean["time_rank"].nunique() < 2:
        continue
    slope, *_ = stats.linregress(sub_mean["time_rank"], sub_mean["amplitude"])
    slopes.append(slope)

w_stat, p_val = stats.wilcoxon(slopes)
print(f"Per-mouse amplitude-vs-pulse-time slopes (n={len(slopes)} mice): mean={np.mean(slopes):.4f}, Wilcoxon p={p_val:.4f}")

## 6. Summary figure for manuscript

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(spu.TEXTWIDTH * 1.5, spu.TEXTWIDTH * 0.65))

_, _, _, mouse_mat = kernels[("residual", "stimulus_onset")]
plot_kernel_panel(axs[0], mouse_mat, "stimulus_onset", "Choice-conditioned kernel\n(control trials, coherence-subtracted)")

summary = amp_df.groupby(["pulse_time_bin", "subject_id"])["amplitude"].mean().reset_index()
order = ["early", "middle", "late"]
for subj, sub in summary.groupby("subject_id"):
    sub = sub.set_index("pulse_time_bin").reindex(order)
    axs[1].plot(order, sub["amplitude"], color="grey", alpha=0.4, marker="o", ms=3)
group_mean = summary.groupby("pulse_time_bin")["amplitude"].mean().reindex(order)
group_sem = summary.groupby("pulse_time_bin")["amplitude"].sem().reindex(order)
axs[1].errorbar(order, group_mean, yerr=group_sem, color="#d11149", lw=2.5, marker="o", capsize=4, label="group mean \u00b1 SEM")
axs[1].set_title("Pulse-triggered kernel amplitude\nvs. pulse timing", fontsize=11)
axs[1].set_xlabel("Pulse timing")
axs[1].set_ylabel("Peak |residual ME| (0-200ms post-pulse)")
axs[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "motion_energy_kernel_summary.png", dpi=300)
plt.savefig(FIGURE_DIR / "motion_energy_kernel_summary.svg")
plt.show()

## Caveats / things to verify before trusting these results

- **Not executed.** Written against the schema used in `2.10`-`2.13` but the data directory
  (`/mnt/cnc_backup/vaib_data/mouse-pulse-experiment`) was not mounted in the environment this was
  authored in. Run top-to-bottom and sanity-check shapes/magnitudes at each step.
- **`choice` encoding** is assumed to be `1 = rightward, 0 = leftward` (matches the commented-out
  alternative in `2.13`). Confirm against `notebooks/0.1-preprocessing-data.ipynb` -- if reversed, every
  kernel and the cohort/pulse-timing plots will be sign-flipped.
- **Frame rate mismatch across rigs.** UCLA runs at 190Hz, UW effectively at 60Hz (`scripts/ucla_rdk.py`
  vs `scripts/uw_rdk.py`). `FRAME_MS = 1000/60` is applied uniformly here; confirm whether
  `motion_energies_normalized/me_{type}.pkl` was already resampled to a common rate in `2.10` -- if not,
  the x-axis (and the `window_frames`/`onset_frame` arithmetic in the pulse-aligned kernel) will be wrong
  for UCLA trials specifically, which would also bias the cohort-replication comparison in section 4.
- **NaN thresholds, window sizes, and trim thresholds** (`nan_threshold`, `trim_threshold=0.4`,
  `window_frames=60`, the 200/300ms amplitude windows) are provisional choices, not fit to the data.
  Re-tune against the actual RT distribution and pulse-onset timing once real data is loaded.
- **Cluster-permutation test** is a minimal from-scratch sign-flip implementation, not a
  peer-reviewed-library implementation -- fine for exploration, but consider `mne.stats` for the version
  that ends up in a manuscript.
- **Cohort comparison (Section 4)** has only ~5 mice per cohort -- likely underpowered for anything beyond
  a rough sanity check; report it as a replication check, not a formal equivalence test.
- Trials are filtered to `reaction_time < 3000` ms (ms, after the `*1000` conversion) to match `2.13`;
  revisit against the `main.yaml` acceptance criterion (`max_reaction_time`, currently 5s) if you want the
  kernel to use the full valid-trial range.